# 4. Extract phase features

Versione pulita: usa setup unico del progetto. Gli output dati restano in `data/processed/features`.

In [1]:
from pathlib import Path
import sys

def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() or (candidate / ".git").exists() or (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Project root not found. Run the notebook from inside the project repository.")

def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path

def report_dir(root: Path, analysis_name: str, subfolder: str | None = None) -> Path:
    base = root / "reports" / analysis_name
    return ensure_dir(base if subfolder is None else base / subfolder)

ROOT = find_project_root()
print("ROOT:", ROOT)


ROOT: /home/eleonorab/repository/convective_cooling


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd



ROOT = find_project_root()
INPUT_FILE = ROOT / "data" / "processed" / "analysis" / "analysis_dataset_wide.csv"
OUTPUT_DIR = ROOT / "data" / "processed" / "features"
ensure_dir(OUTPUT_DIR)

PHASE_ORDER = [
    "standing_1",
    "walk_low_1",
    "walk_mod_1",
    "walk_low_2",
    "walk_mod_2",
    "standing_2",
]

SKIN_TEMP_SIGNALS = [
    "t_neck",
    "t_scapula",
    "t_hand",
    "t_shin",
]


def load_data() -> pd.DataFrame:
    if not INPUT_FILE.exists():
        raise FileNotFoundError(f"Missing input file: {INPUT_FILE}")

    df = pd.read_csv(INPUT_FILE)

    numeric_cols = [
        "elapsed_minutes",
        "elapsed_seconds",
        *SKIN_TEMP_SIGNALS,
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "condition" in df.columns:
        df["condition"] = df["condition"].astype(str).str.strip()
    if "subject_id" in df.columns:
        df["subject_id"] = df["subject_id"].astype(str).str.strip()
    if "phase" in df.columns:
        df["phase"] = df["phase"].astype(str).str.strip()

    return df


def add_mean_skin_temperature(df: pd.DataFrame) -> pd.DataFrame:
    available = [c for c in SKIN_TEMP_SIGNALS if c in df.columns]
    if not available:
        raise ValueError("None of the skin temperature signals are available.")

    df = df.copy()
    df["mean_skin_temperature"] = df[available].mean(axis=1, skipna=True)
    return df


def get_signal_columns(df: pd.DataFrame) -> list[str]:
    exclude = {
        "subject_id",
        "condition",
        "timestamp",
        "elapsed_seconds",
        "elapsed_minutes",
        "phase",
        "time_bin_5min",
    }
    return [c for c in df.columns if c not in exclude]


def compute_baselines(df: pd.DataFrame, signal_cols: list[str]) -> pd.DataFrame:
    # ultimi 2 minuti di standing_1: 8 <= t < 10
    baseline_window = df[
        (df["phase"] == "standing_1") &
        (df["elapsed_minutes"] >= 8) &
        (df["elapsed_minutes"] < 10)
    ].copy()

    rows = []
    for (subject_id, condition), sub in baseline_window.groupby(["subject_id", "condition"], observed=True):
        for signal in signal_cols:
            if signal not in sub.columns:
                continue
            val = pd.to_numeric(sub[signal], errors="coerce").mean()
            rows.append({
                "subject_id": subject_id,
                "condition": condition,
                "signal": signal,
                "baseline_value": val,
            })

    return pd.DataFrame(rows)


def compute_slope(x: np.ndarray, y: np.ndarray) -> float:
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    if len(x) < 2:
        return np.nan

    coeffs = np.polyfit(x, y, 1)
    return coeffs[0]


def extract_phase_features(df: pd.DataFrame, signal_cols: list[str], baselines: pd.DataFrame) -> pd.DataFrame:
    baseline_lookup = baselines.set_index(["subject_id", "condition", "signal"])["baseline_value"].to_dict()

    rows = []

    grouped = df.groupby(["subject_id", "condition", "phase"], observed=True)

    for (subject_id, condition, phase), sub in grouped:
        sub = sub.sort_values("elapsed_minutes").copy()

        for signal in signal_cols:
            if signal not in sub.columns:
                continue

            y = pd.to_numeric(sub[signal], errors="coerce")
            x = pd.to_numeric(sub["elapsed_minutes"], errors="coerce")

            valid = pd.DataFrame({"x": x, "y": y}).dropna()
            if valid.empty:
                continue

            first_val = valid["y"].iloc[0]
            last_val = valid["y"].iloc[-1]
            delta_in_out = last_val - first_val
            slope = compute_slope(valid["x"].to_numpy(), valid["y"].to_numpy())

            baseline_value = baseline_lookup.get((subject_id, condition, signal), np.nan)
            mean_val = valid["y"].mean()

            rows.append({
                "subject_id": subject_id,
                "condition": condition,
                "phase": phase,
                "signal": signal,
                "n_samples": len(valid),
                "mean": mean_val,
                "std": valid["y"].std(),
                "median": valid["y"].median(),
                "min": valid["y"].min(),
                "max": valid["y"].max(),
                "delta_in_out": delta_in_out,
                "slope": slope,
                "baseline_value": baseline_value,
                "delta_mean_from_baseline": mean_val - baseline_value if pd.notna(baseline_value) else np.nan,
            })

    out = pd.DataFrame(rows)
    if not out.empty:
        out["phase"] = pd.Categorical(out["phase"], categories=PHASE_ORDER, ordered=True)
        out = out.sort_values(["signal", "condition", "subject_id", "phase"]).reset_index(drop=True)

    return out


def main():
    df = load_data()
    df = add_mean_skin_temperature(df)

    signal_cols = get_signal_columns(df)

    baselines = compute_baselines(df, signal_cols)
    phase_features = extract_phase_features(df, signal_cols, baselines)

    df.to_csv(OUTPUT_DIR / "analysis_dataset_with_tsk.csv", index=False)
    baselines.to_csv(OUTPUT_DIR / "baseline_values.csv", index=False)
    phase_features.to_csv(OUTPUT_DIR / "phase_features_long.csv", index=False)

    print("Saved:")
    print(OUTPUT_DIR / "analysis_dataset_with_tsk.csv")
    print(OUTPUT_DIR / "baseline_values.csv")
    print(OUTPUT_DIR / "phase_features_long.csv")


if __name__ == "__main__":
    main()

Saved:
/home/eleonorab/repository/convective_cooling/data/processed/features/analysis_dataset_with_tsk.csv
/home/eleonorab/repository/convective_cooling/data/processed/features/baseline_values.csv
/home/eleonorab/repository/convective_cooling/data/processed/features/phase_features_long.csv


In [5]:
from pathlib import Path
import pandas as pd
import statsmodels.formula.api as smf

CSV_PATH = Path(
    "/home/eleonorab/repository/convective_cooling/reports/"
    "04_mechanistic_analysis/convective_indices/"
    "convective_indices_subject_phase_summary.csv"
)

OUT_DIR = CSV_PATH.parent

df = pd.read_csv(CSV_PATH)

df = df.dropna(subset=["subject_id", "condition", "phase", "mean_deltaT_loc"]).copy()

df["condition"] = pd.Categorical(
    df["condition"],
    categories=["no_wind", "wind_low", "wind_high"],
    ordered=True,
)

df["phase"] = pd.Categorical(
    df["phase"],
    categories=[
        "standing_1",
        "walk_low_1",
        "walk_mod_1",
        "walk_low_2",
        "walk_mod_2",
        "standing_2",
    ],
    ordered=True,
)

df = df.dropna(subset=["condition", "phase"]).copy()

print("Data shape:", df.shape)
print("\nCounts by condition and phase:")
print(pd.crosstab(df["phase"], df["condition"]))

# Mixed model
model = smf.mixedlm(
    "mean_deltaT_loc ~ condition * phase",
    data=df,
    groups=df["subject_id"],
)

result = model.fit(
    method="powell",
    reml=False,
    maxiter=2000,
    disp=False,
)

print(result.summary())

coef_table = pd.DataFrame({
    "term": result.params.index,
    "estimate": result.params.values,
    "p_value": result.pvalues.values,
    "ci_low": result.conf_int()[0].values,
    "ci_high": result.conf_int()[1].values,
})

coef_table = coef_table[coef_table["term"] != "Group Var"].copy()

sig_terms = coef_table[coef_table["p_value"] < 0.05].copy()
sig_terms = sig_terms.sort_values("p_value")

coef_table.to_csv(OUT_DIR / "lmm_deltaTloc_coefficients.csv", index=False)
sig_terms.to_csv(OUT_DIR / "lmm_deltaTloc_significant_terms.csv", index=False)

print("\nSignificant terms:")
print(sig_terms)

print("\nSaved:")
print(OUT_DIR / "lmm_deltaTloc_coefficients.csv")
print(OUT_DIR / "lmm_deltaTloc_significant_terms.csv")

from scipy.stats import ttest_ind

for phase in df["phase"].cat.categories:

    no_wind = df.loc[
        (df["phase"] == phase)
        & (df["condition"] == "no_wind"),
        "mean_deltaT_loc",
    ]

    wind_low = df.loc[
        (df["phase"] == phase)
        & (df["condition"] == "wind_low"),
        "mean_deltaT_loc",
    ]

    wind_high = df.loc[
        (df["phase"] == phase)
        & (df["condition"] == "wind_high"),
        "mean_deltaT_loc",
    ]

    print("\n" + "=" * 80)
    print(phase)

    print(
        "wind_low vs no_wind:",
        ttest_ind(wind_low, no_wind, equal_var=False)
    )

    print(
        "wind_high vs no_wind:",
        ttest_ind(wind_high, no_wind, equal_var=False)
    )

Data shape: (132, 24)

Counts by condition and phase:
condition   no_wind  wind_low  wind_high
phase                                   
standing_1        8         7          7
walk_low_1        8         7          7
walk_mod_1        8         7          7
walk_low_2        8         7          7
walk_mod_2        8         7          7
standing_2        8         7          7
                        Mixed Linear Model Regression Results
Model:                    MixedLM         Dependent Variable:         mean_deltaT_loc
No. Observations:         132             Method:                     ML             
No. Groups:               8               Scale:                      0.5884         
Min. group size:          12              Log-Likelihood:             -155.0812      
Max. group size:          18              Converged:                  Yes            
Mean group size:          16.5                                                       
----------------------------------------